# MLP-Variants Research Quickstart (Colab / Kaggle / Local)

Welcome to the **MLP-Variants** experimentation foundation repository.
This notebook demonstrates how to:
1. Setup environment on Google Colab, Kaggle, or Local machine
2. Set deterministic seeds across PyTorch, NumPy, and DataLoader workers
3. Build datasets (CIFAR-100, Tiny-ImageNet, ImageNet subset)
4. Instantiate and inspect the MLP-Mixer reference architecture
5. Train interactively using the modular `Trainer` class
6. Plot loss curves and evaluate saved checkpoints

## 1. Environment & Path Setup (Run on Colab / Kaggle)
If running in Google Colab or Kaggle, clone the repo (if not already cloned) and install editable package.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Project root added to sys.path: {REPO_ROOT}")

# Check device
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Seed Control & Determinism
To ensure experiments are completely reproducible across branches and machines, initialize seeds:

In [ ]:
from src.utils.seed import set_seed

SEED = 42
set_seed(seed=SEED, deterministic=True)
print(f"Deterministic seed {SEED} configured across random, numpy, torch, and CUDA.")

## 3. Configuration Management
Load configuration from YAML and inspect parameters:

In [ ]:
from src.config.parser import load_config

# Load baseline CIFAR-100 config
cfg = load_config(
    config_path=REPO_ROOT / "configs/cifar100_mlp_mixer.yaml",
    default_config_path=REPO_ROOT / "configs/default.yaml",
)

# Override training epochs and batch size for interactive demonstration
cfg.training.epochs = 3
cfg.data.batch_size = 64
cfg.training.lr = 1e-3

print("Loaded Configuration:")
print(f"Dataset:     {cfg.data.dataset}")
print(f"Model:       {cfg.model.name}")
print(f"Epochs:      {cfg.training.epochs}")
print(f"Batch size:  {cfg.data.batch_size}")
print(f"Learning rate: {cfg.training.lr}")

## 4. Build Data Pipelines
Instantiate DataLoaders using the unified `build_dataloaders` interface:

In [ ]:
from src.data import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(cfg)
print(f"Train batches: {len(train_loader)} ({len(train_loader.dataset)} samples)")
print(f"Val batches:   {len(val_loader)} ({len(val_loader.dataset)} samples)")
print(f"Test batches:  {len(test_loader)} ({len(test_loader.dataset)} samples)")

# Inspect a batch
sample_images, sample_targets = next(iter(train_loader))
print(f"Batch image shape:  {sample_images.shape}")
print(f"Batch target shape: {sample_targets.shape}")

## 5. Instantiate Model Architecture
Build the registered `mlp_mixer` (or your teammate's custom variant) via `build_model`:

In [ ]:
from src.models import build_model

model = build_model(cfg)
print(f"Model Type: {type(model).__name__}")
print(f"Total Parameters:     {model.num_parameters(trainable_only=False):,}")
print(f"Trainable Parameters: {model.num_parameters(trainable_only=True):,}")

# Verify forward pass shape
with torch.no_grad():
    dummy_out = model(sample_images[:4])
    print(f"Forward pass successful! Output logits shape: {dummy_out.shape}")

## 6. Setup Experiment Directory & Train
Generate a standardized experiment directory with automatic git metadata tracking and train:

In [ ]:
from src.utils.naming import generate_experiment_name, setup_experiment_dir
from src.engine.trainer import Trainer

exp_name = generate_experiment_name(
    model_name=cfg.model.name,
    dataset_name=cfg.data.dataset,
    tag="notebook_demo",
    seed=SEED,
)
exp_dirs = setup_experiment_dir(exp_name, base_dir=REPO_ROOT / "runs")
print(f"Experiment artifacts dir: {exp_dirs['root']}")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    exp_dirs=exp_dirs,
)

# Run training for specified epochs
results = trainer.fit(epochs=cfg.training.epochs)
print(f"Training finished! Best Validation Acc@1: {results['best_top1']:.2f}%")

## 7. Visualize Learning Curves

In [ ]:
import matplotlib.pyplot as plt

history = results["history"]
epochs = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss = [h["val_loss"] for h in history]
val_top1 = [h["val_top1"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, label="Train Loss", marker="o")
ax1.plot(epochs, val_loss, label="Val Loss", marker="s")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("CrossEntropy Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, val_top1, label="Val Top-1 Acc (%)", color="green", marker="^")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Validation Accuracy")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()